In [ ]:
import os

# Get the current working directory
cwd = os.getcwd()

print("Current working directory:", cwd)

Current working directory: /home/svs25/SAE/data


: 

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

from pathlib import Path
from typing import List, Tuple

import torch
import pandas as pd
from tqdm import tqdm

from progen3.modeling import ProGen3ForCausalLM
from progen3.batch_preparer import ProGen3BatchPreparer


# =========================================================
# Config
# =========================================================
MODEL_NAME = "Profluent-Bio/progen3-112m"
DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"

TRAIN_PATH = Path("processed/s1a70/s1a70_train.csv")
VAL_PATH   = Path("processed/s1a70/s1a70_val.csv")
TEST_PATH  = Path("processed/s1a70/s1a70_test.csv")

OUTPUT_DIR = Path("embeddings")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SEQ_COL = "sequence"

# hidden_states[0] = embedding output
# hidden_states[1] = transformer block 1 output
# ...
# hidden_states[6] = transformer block 6 output
TARGET_HIDDEN_INDEX = 6

BATCH_SIZE = 8
MODEL_DTYPE = torch.bfloat16 if torch.cuda.is_available() else torch.float32
SAVE_DTYPE = torch.float16

# Set True once if you want to inspect batch keys and stop
DEBUG_BATCH_KEYS = False


# =========================================================
# Data loading
# =========================================================
def load_sequences(csv_path: Path, seq_col: str = "sequence") -> Tuple[pd.DataFrame, List[str]]:
    df = pd.read_csv(csv_path)
    if seq_col not in df.columns:
        raise ValueError(
            f"Column '{seq_col}' not found in {csv_path}. "
            f"Available columns: {list(df.columns)}"
        )
    sequences = df[seq_col].astype(str).tolist()
    return df, sequences


# =========================================================
# Mask inference
# =========================================================
def infer_attention_mask(batch_kwargs: dict, model, hidden: torch.Tensor) -> torch.Tensor:
    """
    Construct an attention mask for valid tokens.

    Priority:
    1. Use batch_kwargs['attention_mask'] if present
    2. Infer from input_ids and pad_token_id if available
    3. Fall back to all-ones mask matching hidden[:2] shape
    """
    if "attention_mask" in batch_kwargs:
        return batch_kwargs["attention_mask"]

    if "input_ids" in batch_kwargs:
        input_ids = batch_kwargs["input_ids"]
        pad_token_id = getattr(model.config, "pad_token_id", None)

        if pad_token_id is not None:
            return (input_ids != pad_token_id).long()

        return torch.ones_like(input_ids, dtype=torch.long)

    # Fallback: assume all returned positions are valid
    B, T, _ = hidden.shape
    return torch.ones((B, T), dtype=torch.long, device=hidden.device)


# =========================================================
# Model forward
# =========================================================
@torch.no_grad()
def get_batch_hidden_states(
    model: ProGen3ForCausalLM,
    batch_preparer: ProGen3BatchPreparer,
    sequences: List[str],
    device: str,
    target_hidden_index: int,
):
    batch_kwargs = batch_preparer.get_batch_kwargs(
        sequences,
        device=device,
        reverse=False,
    )

    if DEBUG_BATCH_KEYS:
        print("batch_kwargs keys:", list(batch_kwargs.keys()))
        for k, v in batch_kwargs.items():
            if torch.is_tensor(v):
                print(f"{k}: shape={tuple(v.shape)}, dtype={v.dtype}, device={v.device}")
            else:
                print(f"{k}: type={type(v)}")
        raise SystemExit("Stopping after batch inspection because DEBUG_BATCH_KEYS=True")

    outputs = model(
        **batch_kwargs,
        return_dict=True,
        output_hidden_states=True,
        use_cache=False,
    )

    hidden_states = outputs.hidden_states
    if hidden_states is None:
        raise RuntimeError("Model did not return hidden_states.")

    if target_hidden_index >= len(hidden_states):
        raise IndexError(
            f"Requested hidden state index {target_hidden_index}, "
            f"but model returned only {len(hidden_states)} hidden states."
        )

    hidden = hidden_states[target_hidden_index]  # [B, T, D]
    attention_mask = infer_attention_mask(batch_kwargs, model, hidden)

    return hidden, attention_mask


# =========================================================
# Flatten tokens
# =========================================================
def flatten_valid_tokens(
    hidden: torch.Tensor,          # [B, T, D]
    attention_mask: torch.Tensor,  # [B, T]
    batch_seq_ids: List[int],
):
    token_vecs = []
    seq_ids = []
    token_positions = []

    B, T, D = hidden.shape

    for i in range(B):
        valid_pos = attention_mask[i].bool().nonzero(as_tuple=False).squeeze(-1)

        if valid_pos.numel() == 0:
            continue

        valid_hidden = hidden[i, valid_pos]  # [L_i, D]
        token_vecs.append(valid_hidden)

        seq_ids.append(
            torch.full((valid_hidden.shape[0],), batch_seq_ids[i], dtype=torch.long)
        )
        token_positions.append(valid_pos.to(torch.long))

    if len(token_vecs) == 0:
        return (
            torch.empty((0, D), dtype=hidden.dtype),
            torch.empty((0,), dtype=torch.long),
            torch.empty((0,), dtype=torch.long),
        )

    flat_hidden = torch.cat(token_vecs, dim=0)
    flat_seq_ids = torch.cat(seq_ids, dim=0)
    flat_token_positions = torch.cat(token_positions, dim=0)

    return flat_hidden, flat_seq_ids, flat_token_positions


# =========================================================
# Process one split
# =========================================================
def process_split(
    split_csv: Path,
    split_name: str,
    model: ProGen3ForCausalLM,
    batch_preparer: ProGen3BatchPreparer,
    batch_size: int,
    target_hidden_index: int,
    device: str,
    save_dtype: torch.dtype,
    output_dir: Path,
):
    print(f"\nLoading {split_name} split from {split_csv}")
    df, sequences = load_sequences(split_csv, seq_col=SEQ_COL)

    print(f"{split_name}: {len(sequences)} sequences")
    if len(sequences) > 0:
        print(f"First sequence length (AA chars): {len(sequences[0])}")

    all_hidden = []
    all_seq_ids = []
    all_token_positions = []
    all_lengths = []

    for start in tqdm(range(0, len(sequences), batch_size), desc=f"{split_name} batches"):
        end = min(start + batch_size, len(sequences))
        batch_seqs = sequences[start:end]
        batch_seq_ids = list(range(start, end))

        hidden, attention_mask = get_batch_hidden_states(
            model=model,
            batch_preparer=batch_preparer,
            sequences=batch_seqs,
            device=device,
            target_hidden_index=target_hidden_index,
        )

        hidden = hidden.detach().cpu()
        attention_mask = attention_mask.detach().cpu()

        flat_hidden, flat_seq_ids, flat_token_positions = flatten_valid_tokens(
            hidden=hidden,
            attention_mask=attention_mask,
            batch_seq_ids=batch_seq_ids,
        )

        lengths = attention_mask.sum(dim=1).to(torch.long)

        all_hidden.append(flat_hidden.to(save_dtype))
        all_seq_ids.append(flat_seq_ids)
        all_token_positions.append(flat_token_positions)
        all_lengths.append(lengths)

        if device.startswith("cuda"):
            torch.cuda.empty_cache()

    if len(all_hidden) == 0:
        raise RuntimeError(f"No activations were collected for split '{split_name}'.")

    activations = torch.cat(all_hidden, dim=0)
    sequence_ids = torch.cat(all_seq_ids, dim=0)
    token_positions = torch.cat(all_token_positions, dim=0)
    lengths = torch.cat(all_lengths, dim=0)

    save_path = output_dir / f"s1a70_{split_name}_layer{target_hidden_index}_raw.pt"
    torch.save(
        {
            "activations": activations,          # [N_tokens, D]
            "sequence_ids": sequence_ids,        # [N_tokens]
            "token_positions": token_positions,  # [N_tokens]
            "lengths": lengths,                  # [N_sequences]
            "split_name": split_name,
            "source_csv": str(split_csv),
            "sequence_column": SEQ_COL,
            "model_name": MODEL_NAME,
            "hidden_index": target_hidden_index,
            "save_dtype": str(save_dtype),
        },
        save_path,
    )

    print(f"Saved {split_name} split to: {save_path}")
    print(f"  activations shape: {tuple(activations.shape)}")
    print(f"  sequence_ids shape: {tuple(sequence_ids.shape)}")
    print(f"  token_positions shape: {tuple(token_positions.shape)}")
    print(f"  lengths shape: {tuple(lengths.shape)}")


# =========================================================
# Main
# =========================================================
def main():
    print(f"Loading model: {MODEL_NAME}")
    print(f"Using device: {DEVICE}")
    print(f"CUDA_VISIBLE_DEVICES: {os.environ.get('CUDA_VISIBLE_DEVICES')}")

    model = ProGen3ForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=MODEL_DTYPE,
    ).to(DEVICE)
    model.eval()

    if torch.cuda.is_available():
        print(f"Current CUDA device index: {torch.cuda.current_device()}")
        print(f"GPU name: {torch.cuda.get_device_name(torch.cuda.current_device())}")

    batch_preparer = ProGen3BatchPreparer()

    # Optional one-time hidden-state sanity check
    with torch.no_grad():
        test_inputs = batch_preparer.get_batch_kwargs(
            ["MALWMRLLPLLALLALWGPDPAAAFVNQHLCGSHLVEALYLVCGERGFFYTPKTRREAEDLQVGQVELGGGPGAGSLQPLALEGSLQKRGIVEQCCTSICSLYQLENYCN"],
            device=DEVICE,
            reverse=False,
        )
        test_outputs = model(
            **test_inputs,
            return_dict=True,
            output_hidden_states=True,
            use_cache=False,
        )
        print(f"Model has {model.config.num_hidden_layers} transformer layers")
        print(f"hidden_states tuple length: {len(test_outputs.hidden_states)}")
        print(f"Layer {TARGET_HIDDEN_INDEX} hidden shape: {tuple(test_outputs.hidden_states[TARGET_HIDDEN_INDEX].shape)}")

    process_split(
        split_csv=TRAIN_PATH,
        split_name="train",
        model=model,
        batch_preparer=batch_preparer,
        batch_size=BATCH_SIZE,
        target_hidden_index=TARGET_HIDDEN_INDEX,
        device=DEVICE,
        save_dtype=SAVE_DTYPE,
        output_dir=OUTPUT_DIR,
    )

    process_split(
        split_csv=VAL_PATH,
        split_name="val",
        model=model,
        batch_preparer=batch_preparer,
        batch_size=BATCH_SIZE,
        target_hidden_index=TARGET_HIDDEN_INDEX,
        device=DEVICE,
        save_dtype=SAVE_DTYPE,
        output_dir=OUTPUT_DIR,
    )

    process_split(
        split_csv=TEST_PATH,
        split_name="test",
        model=model,
        batch_preparer=batch_preparer,
        batch_size=BATCH_SIZE,
        target_hidden_index=TARGET_HIDDEN_INDEX,
        device=DEVICE,
        save_dtype=SAVE_DTYPE,
        output_dir=OUTPUT_DIR,
    )

    print("\nDone.")


if __name__ == "__main__":
    main()

/home/svs25/.conda/envs/GRPO/lib/python3.12/site-packages/flash_attn/ops/triton/layer_norm.py:984: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @custom_fwd
/home/svs25/.conda/envs/GRPO/lib/python3.12/site-packages/flash_attn/ops/triton/layer_norm.py:1043: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
  @custom_bwd


Loading model: Profluent-Bio/progen3-112m
Using device: cuda:0
CUDA_VISIBLE_DEVICES: 0
Current CUDA device index: 0
GPU name: NVIDIA L40S
Model has 10 transformer layers
hidden_states tuple length: 11
Layer 6 hidden shape: (1, 114, 384)

Loading train split from processed/s1a70/s1a70_train.csv
train: 38318 sequences
First sequence length (AA chars): 269


train batches: 100%|██████████| 4790/4790 [01:52<00:00, 42.64it/s]


Saved train split to: embeddings/s1a70_train_layer6_raw.pt
  activations shape: (10271296, 384)
  sequence_ids shape: (10271296,)
  token_positions shape: (10271296,)
  lengths shape: (38318,)

Loading val split from processed/s1a70/s1a70_val.csv
val: 8354 sequences
First sequence length (AA chars): 264


val batches: 100%|██████████| 1045/1045 [00:24<00:00, 42.26it/s]


Saved val split to: embeddings/s1a70_val_layer6_raw.pt
  activations shape: (2238702, 384)
  sequence_ids shape: (2238702,)
  token_positions shape: (2238702,)
  lengths shape: (8354,)

Loading test split from processed/s1a70/s1a70_test.csv
test: 7963 sequences
First sequence length (AA chars): 248


test batches: 100%|██████████| 996/996 [00:23<00:00, 42.45it/s]


Saved test split to: embeddings/s1a70_test_layer6_raw.pt
  activations shape: (2134528, 384)
  sequence_ids shape: (2134528,)
  token_positions shape: (2134528,)
  lengths shape: (7963,)

Done.


In [4]:
import torch

x = torch.load("embeddings/s1a70_train_layer6_raw.pt")
print(x.keys())
print(x["activations"].shape)
print(x["sequence_ids"].shape)
print(x["token_positions"].shape)
print(x["lengths"][:10])
print(x["activations"].dtype)
print(x["activations"].mean().item(), x["activations"].std().item())
print(x["sequence_ids"].min().item(), x["sequence_ids"].max().item())
print(x["token_positions"].min().item(), x["token_positions"].max().item())
print(x["lengths"].sum().item(), x["activations"].shape[0])

/tmp/ipykernel_609935/3784207336.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  x = torch.load("embeddings/s1a70_train_layer6_raw.pt")


dict_keys(['activations', 'sequence_ids', 'token_positions', 'lengths', 'split_name', 'source_csv', 'sequence_column', 'model_name', 'hidden_index', 'save_dtype'])
torch.Size([10271296, 384])
torch.Size([10271296])
torch.Size([10271296])
tensor([273, 273, 271, 251, 257, 248, 321, 264, 251, 279])
torch.float16
-0.216064453125 28.375
0 38317
0 323
10271296 10271296


In [5]:
seq_id = 0
mask = x["sequence_ids"] == seq_id
print(mask.sum().item(), x["lengths"][seq_id].item())
print(x["token_positions"][mask][:20])
print(x["token_positions"][mask][-10:])

273 273
tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
        18, 19])
tensor([263, 264, 265, 266, 267, 268, 269, 270, 271, 272])
